# Lab 2.2 — Build the retriever bench

**Before you start:** select **Cell > Run All** to initialize the harness.

Implement four retriever strategies, measure each on the dev query set, then save `retriever.json`.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from ara_metrics import (
    retriever_bm25, retriever_dense, retriever_hybrid_rrf, retriever_hybrid_rerank,
    run_queries_with_template, ndcg_at_k, precision_at_k, p50 as p50_latency,
)

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

from elasticsearch import Elasticsearch
ES_URL    = os.environ['ES_URL']
ES_KEY    = os.environ['ES_API_KEY']
EMBED_ID  = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')
RERANK_ID = os.environ.get('ARA_RERANK_ID', '.jina-reranker-v3')

es = Elasticsearch(ES_URL, api_key=ES_KEY, request_timeout=60)
INDEX = 'cortex-corpus-live'

dev_queries = json.loads(pathlib.Path('/home/elastic/dev-sets/dev-queries.json').read_text())
print(f'Harness ready. EMBED_ID={EMBED_ID}, RERANK_ID={RERANK_ID}')
print(f'Index: {INDEX}  |  Dev queries: {len(dev_queries)}')

---
## Step 1 — Implement retriever templates

Each template is a function that takes a query string and returns an Elasticsearch retriever body.

| Key | Strategy | Body shape |
|---|---|---|
| `bm25` | `match` on `body_text` | `{"standard": {"query": {"match": ...}}}` |
| `dense` | `semantic` on `body` | `{"standard": {"query": {"semantic": ...}}}` |
| `hybrid` | RRF fusion | `retriever_hybrid_rrf(...)` |
| `hybrid_rerank` | Reranked hybrid | `retriever_hybrid_rerank(...)` |

In [ ]:
# ── YOUR WORK ── Define four retriever templates ─────────────────────────────
# Each template is a dict with {query_text} placeholders.
# ara_metrics provides builders for hybrid and rerank templates.

# BM25: match on body_text (replace {query_text} with the search term)
MY_BM25 = {
    # YOUR CODE HERE — use retriever_bm25() or write it manually
    # Example: retriever_bm25(field="body_text")
}

# Dense: semantic search on body field
MY_DENSE = {
    # YOUR CODE HERE — use retriever_dense() or write it manually
}

# Hybrid: RRF fusion of BM25 + dense
MY_HYBRID = retriever_hybrid_rrf(
    # YOUR CODE HERE — set text_field and semantic_field
    # text_field="body_text", semantic_field="body"
)

# Hybrid + rerank: text_similarity_reranker on top of hybrid
MY_HYBRID_RERANK = retriever_hybrid_rerank(
    # YOUR CODE HERE — first arg is rerank_id, then field names
    # RERANK_ID, text_field="body_text", semantic_field="body", rank_window_size=20
)

In [ ]:
# ── Run each template against the dev query set ──────────────────────────────
# run_queries_with_template runs one template at a time.
# n_passes=3 measures latency with multiple runs.

templates = {
    'bm25': MY_BM25,
    'dense': MY_DENSE,
    'hybrid': MY_HYBRID,
    'hybrid_rerank': MY_HYBRID_RERANK,
}

all_results = {}
for name, tmpl in templates.items():
    passes = 1 if name != 'hybrid_rerank' else 3
    results = run_queries_with_template(es, INDEX, dev_queries, tmpl, k=5, n_passes=passes)
    all_results[name] = results
    ndcg = ndcg_at_k(results, dev_queries, k=5)
    prec = precision_at_k(results, dev_queries, k=5)
    lat  = p50_latency(results)
    print(f"{name:20s}  nDCG@5={ndcg:.3f}  P@5={prec:.3f}  p50={lat:.0f}ms")

In [ ]:
# ── Save retriever.json (run after implementing templates above) ──────────────
retriever_json = {
    'bm25':          MY_BM25,
    'dense':         MY_DENSE,
    'hybrid':        MY_HYBRID,
    'hybrid_rerank': MY_HYBRID_RERANK,
}

pathlib.Path('/home/elastic/retriever.json').write_text(json.dumps(retriever_json, indent=2))
print('retriever.json saved. Select Check in the sidebar.')